In [2]:
import os
from dotenv import load_dotenv

load_dotenv()

API_KEY = os.getenv("EMBER_API_KEY")

print(API_KEY is not None)
print(len(API_KEY) if API_KEY else 0)

True
36


In [4]:
import requests

BASE_URL = "https://api.ember-energy.org"
endpoint = "/v1/electricity-generation/monthly"

params = {
    "entity_code": "DEU",
    "api_key": API_KEY
}

response = requests.get(
    f"{BASE_URL}{endpoint}",
    params=params,
    timeout=30
)

print("Status code:", response.status_code)

Status code: 200


In [5]:
data = response.json()

print("Python type:", type(data))

if isinstance(data, dict):
    print("Top-level keys:", data.keys())

Python type: <class 'dict'>
Top-level keys: dict_keys(['stats', 'data'])


In [6]:
import json

print(json.dumps(data, indent=2)[:3000])


{
  "stats": {
    "timestamp": "2026-09-10T21:01:21.304210Z",
    "response_time_in_seconds": 0.343,
    "rate_limit": "No",
    "number_of_records": 2380,
    "query_parameters_used": {
      "entity_code": [
        "DEU"
      ]
    },
    "available_metrics": [
      "generation_twh",
      "share_of_generation_pct"
    ],
    "query_value_range": {
      "date": {
        "min": "2015-01-01",
        "max": "2026-08-01"
      },
      "generation_twh": {
        "min": -7.06,
        "max": 59.14
      },
      "share_of_generation_pct": {
        "min": -12.15,
        "max": 116.07
      }
    },
    "query_all_dates_value_range": {}
  },
  "data": [
    {
      "entity": "Germany",
      "entity_code": "DEU",
      "is_aggregate_entity": false,
      "date": "2015-01-01",
      "series": "Bioenergy",
      "is_aggregate_series": false,
      "generation_twh": 3.87,
      "share_of_generation_pct": 7.14
    },
    {
      "entity": "Germany",
      "entity_code": "DEU",
      "

In [8]:
import pandas as pd

df = pd.DataFrame(data["data"])

print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

df.head()

Shape: (2380, 8)

Columns:
['entity', 'entity_code', 'is_aggregate_entity', 'date', 'series', 'is_aggregate_series', 'generation_twh', 'share_of_generation_pct']


,entity,entity_code,is_aggregate_entity,date,series,is_aggregate_series,generation_twh,share_of_generation_pct
0,Germany,DEU,False,2015-01-01,Bioenergy,False,3.87,7.14
1,Germany,DEU,False,2015-01-01,Clean,True,24.36,44.98
2,Germany,DEU,False,2015-01-01,Coal,False,21.62,39.92
3,Germany,DEU,False,2015-01-01,Demand,True,49.33,91.10
4,Germany,DEU,False,2015-01-01,Fossil,True,29.79,55.02


In [9]:
series = (
    df[["series", "is_aggregate_series"]]
    .drop_duplicates()
    .sort_values(["is_aggregate_series", "series"])
    .reset_index(drop=True)
)

series

,series,is_aggregate_series
0,Bioenergy,False
1,Coal,False
2,Gas,False
3,Hydro,False
4,Net imports,False
5,Nuclear,False
6,Other fossil,False
7,Other renewables,False
8,Solar,False
9,Wind,False


In [10]:
print("Number of series:", df["series"].nunique())

Number of series: 17


In [11]:
df["date"] = pd.to_datetime(df["date"])

print("Min date:", df["date"].min())
print("Max date:", df["date"].max())
print("Unique months:", df["date"].nunique())

Min date: 2015-01-01 00:00:00
Max date: 2026-08-01 00:00:00
Unique months: 140


In [12]:
df.isna().sum()

entity                     0
entity_code                0
is_aggregate_entity        0
date                       0
series                     0
is_aggregate_series        0
generation_twh             0
share_of_generation_pct    0
dtype: int64